# Image T2T Generator

Schema-driven · Batch-based · Judge-validated

Each Image T2T question has:
- `image_content` — text description of a real-world notice/sign (same format as Image MCQ)
- `question` — an open-ended question about the notice
- `expected_answer` — a model answer (typed response, not multiple choice)
- `explanation` — reasoning behind the expected answer

**CEFR sub-types:**
| Level | Task type |
|---|---|
| A1/A2 | Simple comprehension — "What does the notice say?" |
| B1/B2 | Inference — "What should you do according to the notice?" |
| C1/C2 | Interpretation — "What does this notice imply about the situation?" |

**Output:** `data/image_t2t/image_t2t_generator_output.json`

In [ ]:
from __future__ import annotations
import json, os, re, sys
from pathlib import Path
from typing import Literal, Optional

import dspy
from dotenv import load_dotenv
from pydantic import BaseModel, Field

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env')

from utils import configure_lm
print('Project root:', PROJECT_ROOT)

In [ ]:
lm = configure_lm()
dspy.configure(lm=lm)
print('LM:', lm.model)

In [ ]:
ARTIFACTS           = PROJECT_ROOT / 'artifacts'
DIFFICULTY_ARTIFACT = ARTIFACTS / 'image_mcq_difficulty_optimized.json'
RUBRIC_ARTIFACT     = ARTIFACTS / 'image_mcq_rubric_optimized.json'
OUTPUT_PATH         = PROJECT_ROOT / 'data' / 'image_t2t' / 'image_t2t_generator_output.json'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print('Difficulty artifact:', DIFFICULTY_ARTIFACT.exists())
print('Rubric artifact    :', RUBRIC_ARTIFACT.exists())

In [ ]:
class SubtopicRequirement(BaseModel):
    subtopic:  str
    a1_count:  int = 0
    a2_count:  int = 0
    b1_count:  int = 0
    b2_count:  int = 0
    c1_count:  int = 0
    c2_count:  int = 0

    @property
    def easy_count(self) -> int:   return self.a1_count + self.a2_count
    @property
    def medium_count(self) -> int: return self.b1_count + self.b2_count
    @property
    def hard_count(self) -> int:   return self.c1_count + self.c2_count
    @property
    def total(self) -> int:        return self.easy_count + self.medium_count + self.hard_count


class GenerationConstraints(BaseModel):
    questions_per_iteration: int   = 3
    max_iterations:          int   = 10
    difficulty_match:        bool  = True
    rubric_min_score:        float = 0.7


class InputSchema(BaseModel):
    topic:       str
    subtopics:   list[SubtopicRequirement]
    constraints: GenerationConstraints = Field(default_factory=GenerationConstraints)

In [ ]:
class ImageT2TItem(BaseModel):
    question_number:   int
    topic:             str
    subtopic:          str | None
    target_cefr:       str
    target_difficulty: str
    question_type:     str   # e.g. 'comprehension', 'inference', 'interpretation'
    instruction:       str
    image_content:     str   # text description of the notice/sign
    question:          str
    expected_answer:   str
    explanation:       str


class ImageT2TQuestionStore(BaseModel):
    easy:   list[ImageT2TItem] = Field(default_factory=list)
    medium: list[ImageT2TItem] = Field(default_factory=list)
    hard:   list[ImageT2TItem] = Field(default_factory=list)

    def add(self, item: ImageT2TItem) -> None:
        d = item.target_difficulty.strip().lower()
        if d == 'easy':   self.easy.append(item)
        elif d == 'medium': self.medium.append(item)
        elif d == 'hard':   self.hard.append(item)

    def count(self, difficulty: str) -> int:
        return len(getattr(self, difficulty.lower(), []))

    @property
    def total(self) -> int:
        return len(self.easy) + len(self.medium) + len(self.hard)

In [ ]:
class ExampleImageT2TQuestion(BaseModel):
    instruction:     str
    image_content:   str
    question_type:   str
    question:        str
    expected_answer: str
    explanation:     str
    difficulty:      str
    cefr:            str
    subtopic:        str


class ImageT2TGenerationRequest(BaseModel):
    topic:                       str
    subtopic:                    str
    target_cefr:                 str
    target_difficulty:           str
    question_type:               str
    example_questions:           list[ExampleImageT2TQuestion]
    already_used_image_contents: list[str]
    batch_size:                  int


class ImageT2TGeneratedQuestion(BaseModel):
    instruction:     str
    image_content:   str
    question:        str
    expected_answer: str
    explanation:     str


class ImageT2TGenerationBatch(BaseModel):
    questions: list[ImageT2TGeneratedQuestion]


# Question type per CEFR
_CEFR_QUESTION_TYPE = {
    'A1': 'comprehension',  'A2': 'comprehension',
    'B1': 'inference',      'B2': 'inference',
    'C1': 'interpretation', 'C2': 'interpretation',
}


class ImageT2TGeneratorSignature(dspy.Signature):
    '''Generate a batch of image-based open-answer (T2T) questions for English language learners.

    Each question is based on a real-world notice or sign described in image_content.
    The student reads the notice and writes a typed answer — no multiple-choice options.

    Question types by CEFR:
    - comprehension (A1/A2): "What does the notice say?" / "What is the notice about?"
      expected_answer: 1 short sentence directly quoting or paraphrasing the notice
    - inference (B1/B2): "What should you do according to the notice?" / "What action does the notice require?"
      expected_answer: 1-2 sentences that infer the required action or implication
    - interpretation (C1/C2): "What does this notice imply?" / "Why might this notice be necessary?"
      expected_answer: 2-3 sentences showing deeper understanding of context and implication

    Rules:
    - image_content must be a realistic description of a printed public notice/sign
    - question must be clearly answerable from the image_content alone
    - expected_answer must be a complete sentence, natural English
    - Each image_content must be unique and different from already_used_image_contents
    '''
    request: ImageT2TGenerationRequest = dspy.InputField(desc='Generation parameters and examples')
    batch:   ImageT2TGenerationBatch   = dspy.OutputField(desc='Generated Image T2T questions')

In [ ]:
class DifficultyResult(BaseModel):
    question_id:          str
    predicted_cefr:       Literal['A1', 'A2', 'B1', 'B2', 'C1', 'C2']
    predicted_difficulty: Literal['Easy', 'Medium', 'Hard']

class DifficultyOutput(BaseModel):
    results: list[DifficultyResult]


class ImageT2TDifficultySignature(dspy.Signature):
    '''Classify image-based open-answer questions by CEFR level.
    Consider: vocabulary in image_content, question complexity, depth of expected_answer.
    A1/A2=Easy (comprehension), B1/B2=Medium (inference), C1/C2=Hard (interpretation).
    '''
    questions: str             = dspy.InputField(desc='JSON list of Image T2T questions')
    output:    DifficultyOutput = dspy.OutputField(desc='CEFR and difficulty predictions')


class DifficultyJudgeWrapper:
    def __init__(self, artifact_path: Path):
        self.judge = dspy.Predict(ImageT2TDifficultySignature)
        if artifact_path.exists():
            self.judge.load(str(artifact_path))
        print(f'  DifficultyJudge: {"loaded" if artifact_path.exists() else "zero-shot"}')

    def judge_batch(self, questions: list[dict]) -> list[DifficultyResult]:
        return self.judge(questions=json.dumps(questions)).output.results

In [ ]:
class RubricScore(BaseModel):
    question_id:              str
    image_content_realism:    float = Field(ge=0, le=1)
    question_answerability:   float = Field(ge=0, le=1)
    answer_completeness:      float = Field(ge=0, le=1)
    language_naturalness:     float = Field(ge=0, le=1)
    cefr_appropriateness:     float = Field(ge=0, le=1)
    avg_score:                float = Field(ge=0, le=1)

class RubricOutput(BaseModel):
    scores: list[RubricScore]


class ImageT2TRubricSignature(dspy.Signature):
    '''Score Image T2T question quality on 5 criteria (0-1 each).
    image_content_realism: notice/sign description is believable and realistic
    question_answerability: question can be answered from image_content alone
    answer_completeness: expected_answer is complete and accurate
    language_naturalness: all text is natural English
    cefr_appropriateness: vocabulary and complexity matches the target CEFR
    avg_score: mean of all five scores.
    '''
    questions: str         = dspy.InputField(desc='JSON list of Image T2T questions')
    output:    RubricOutput = dspy.OutputField(desc='Quality scores per question')


class RubricJudgeWrapper:
    def __init__(self, artifact_path: Path):
        self.judge = dspy.Predict(ImageT2TRubricSignature)
        if artifact_path.exists():
            self.judge.load(str(artifact_path))
        print(f'  RubricJudge: {"loaded" if artifact_path.exists() else "zero-shot"}')

    def score_batch(self, questions: list[dict]) -> list[RubricScore]:
        return self.judge(questions=json.dumps(questions)).output.scores

In [ ]:
EXAMPLE_QUESTIONS: list[ExampleImageT2TQuestion] = [
    ExampleImageT2TQuestion(
        instruction='Read the notice and answer the question.',
        image_content='A notice on a classroom door says the room is closed today for cleaning.',
        question_type='comprehension',
        question='What does the notice say about the classroom?',
        expected_answer='The notice says the classroom is closed today for cleaning.',
        explanation='A direct comprehension question answered by reading the notice.',
        difficulty='Easy', cefr='A1', subtopic='public notices'
    ),
    ExampleImageT2TQuestion(
        instruction='Read the notice and answer the question.',
        image_content='A café notice says hot drinks must be consumed inside the café and not taken outside.',
        question_type='comprehension',
        question='What rule does the café notice give about hot drinks?',
        expected_answer='Hot drinks must be consumed inside the café and cannot be taken outside.',
        explanation='A2 comprehension requires paraphrasing the rule stated in the notice.',
        difficulty='Easy', cefr='A2', subtopic='rules and regulations'
    ),
    ExampleImageT2TQuestion(
        instruction='Read the notice and answer the question.',
        image_content='A gym notice says members must book equipment in advance online and bring their membership card.',
        question_type='inference',
        question='What should a gym member do before visiting the gym?',
        expected_answer='A gym member should book equipment online in advance and bring their membership card before visiting.',
        explanation='B1 inference requires combining two requirements from the notice into a practical answer.',
        difficulty='Medium', cefr='B1', subtopic='procedures'
    ),
    ExampleImageT2TQuestion(
        instruction='Read the notice and answer the question in your own words.',
        image_content='A hospital notice states that all visitors must sanitise their hands before entering any ward and that mobile phones must be switched to silent mode.',
        question_type='inference',
        question='What actions are required of visitors entering a hospital ward?',
        expected_answer='Visitors must sanitise their hands and switch their mobile phones to silent mode before entering a ward.',
        explanation='B2 inference requires extracting and combining multiple requirements from a longer notice.',
        difficulty='Medium', cefr='B2', subtopic='health and safety'
    ),
    ExampleImageT2TQuestion(
        instruction='Read the notice and answer the question in your own words.',
        image_content='A conservation area notice warns that the ecosystem is fragile and that removing any natural material, including rocks or plants, is strictly prohibited to preserve biodiversity.',
        question_type='interpretation',
        question='What does this notice suggest about the conservation area, and why might such a rule be necessary?',
        expected_answer='The notice suggests the area contains a delicate ecosystem that could be damaged by human interference. The rule is necessary because removing natural materials, even seemingly insignificant ones, could disrupt biodiversity and harm species that depend on those resources.',
        explanation='C1 interpretation requires understanding the implied purpose and ecological reasoning behind the rule.',
        difficulty='Hard', cefr='C1', subtopic='environment'
    ),
    ExampleImageT2TQuestion(
        instruction='Read the notice carefully and respond in full sentences.',
        image_content='An archive library notice advises researchers that access to primary source documents requires prior written authorisation from the head archivist, and that all materials must be handled with cotton gloves to prevent deterioration.',
        question_type='interpretation',
        question='What does this notice reveal about the nature of the documents held in this library, and what does it imply about the responsibilities of researchers?',
        expected_answer='The notice reveals that the documents are rare, fragile, and of significant historical or academic value, requiring protection from deterioration. It implies that researchers bear a duty of care and must follow strict protocols, reflecting the irreplaceable nature of primary sources and the institution\'s obligation to preserve them for future scholarship.',
        explanation='C2 interpretation demands sophisticated inference about institutional values, researcher responsibilities, and the implicit significance of preservation protocols.',
        difficulty='Hard', cefr='C2', subtopic='academic settings'
    ),
]

In [ ]:
def hard_validate(q: ImageT2TGeneratedQuestion) -> tuple[bool, str]:
    if not q.image_content.strip():
        return False, 'image_content is empty'
    if not q.question.strip():
        return False, 'question is empty'
    if not q.expected_answer.strip():
        return False, 'expected_answer is empty'
    if len(q.expected_answer.split()) < 5:
        return False, f'expected_answer too short ({len(q.expected_answer.split())} words)'
    return True, ''


class ImageT2TGeneratorAgent(dspy.Module):
    def __init__(self, *, store: ImageT2TQuestionStore,
                 difficulty_judge: DifficultyJudgeWrapper,
                 rubric_judge: RubricJudgeWrapper):
        super().__init__()
        self.store            = store
        self.difficulty_judge = difficulty_judge
        self.rubric_judge     = rubric_judge
        self.generator        = dspy.Predict(ImageT2TGeneratorSignature)
        self._counter         = 0

    def forward(self, *, schema, example_questions, topic, subtopic,
                target_cefr, target_difficulty, target_count, rejected, warnings):
        used_contents = [q.image_content for d in (self.store.easy, self.store.medium, self.store.hard)
                         for q in d]
        question_type = _CEFR_QUESTION_TYPE[target_cefr]
        diff_map = {'A1':'Easy','A2':'Easy','B1':'Medium','B2':'Medium','C1':'Hard','C2':'Hard'}
        iteration = 0

        while self.store.count(target_difficulty) < target_count and iteration < schema.constraints.max_iterations:
            iteration += 1
            needed = target_count - self.store.count(target_difficulty)

            req = ImageT2TGenerationRequest(
                topic=topic, subtopic=subtopic,
                target_cefr=target_cefr, target_difficulty=target_difficulty,
                question_type=question_type,
                example_questions=example_questions,
                already_used_image_contents=used_contents,
                batch_size=min(schema.constraints.questions_per_iteration, needed + 2),
            )
            try:
                candidates = self.generator(request=req).batch.questions
            except Exception as e:
                warnings.append(f'[{target_cefr}] iter {iteration}: {e}')
                continue

            valid = []
            for q in candidates:
                ok, reason = hard_validate(q)
                if ok: valid.append(q)
                else:  rejected.append({'cefr': target_cefr, 'reason': reason})

            if not valid: continue

            judge_input = [{'question_id': f'q{i}', 'image_content': q.image_content,
                            'question': q.question, 'expected_answer': q.expected_answer,
                            'target_cefr': target_cefr, 'target_difficulty': target_difficulty}
                           for i, q in enumerate(valid)]

            try:    diff_results = {r.question_id: r for r in self.difficulty_judge.judge_batch(judge_input)}
            except: diff_results = {}
            try:    rubric_results = {r.question_id: r for r in self.rubric_judge.score_batch(judge_input)}
            except: rubric_results = {}

            for i, q in enumerate(valid):
                qid = f'q{i}'
                dr, rr = diff_results.get(qid), rubric_results.get(qid)

                if dr and diff_map.get(dr.predicted_cefr) != target_difficulty:
                    rejected.append({'cefr': target_cefr, 'reason': f'Difficulty mismatch: {dr.predicted_cefr}'})
                    continue
                if rr and rr.avg_score < schema.constraints.rubric_min_score:
                    rejected.append({'cefr': target_cefr, 'reason': f'Rubric low: {rr.avg_score:.2f}'})
                    continue

                self._counter += 1
                self.store.add(ImageT2TItem(
                    question_number=self._counter, topic=topic, subtopic=subtopic,
                    target_cefr=target_cefr, target_difficulty=target_difficulty,
                    question_type=question_type, instruction=q.instruction,
                    image_content=q.image_content, question=q.question,
                    expected_answer=q.expected_answer, explanation=q.explanation,
                ))
                used_contents.append(q.image_content)
                if self.store.count(target_difficulty) >= target_count:
                    break

In [ ]:
_CEFR_TO_DIFFICULTY = {'A1':'Easy','A2':'Easy','B1':'Medium','B2':'Medium','C1':'Hard','C2':'Hard'}
_CEFR_LEVELS = ['A1','A2','B1','B2','C1','C2']

def run_image_t2t_generation(schema: InputSchema) -> tuple[ImageT2TQuestionStore, list, list]:
    difficulty_judge = DifficultyJudgeWrapper(DIFFICULTY_ARTIFACT)
    rubric_judge     = RubricJudgeWrapper(RUBRIC_ARTIFACT)
    store    = ImageT2TQuestionStore()
    rejected, warnings = [], []
    agent = ImageT2TGeneratorAgent(store=store, difficulty_judge=difficulty_judge, rubric_judge=rubric_judge)

    for subtopic_req in schema.subtopics:
        for cefr in _CEFR_LEVELS:
            target_count = getattr(subtopic_req, f'{cefr.lower()}_count', 0)
            if target_count == 0: continue
            difficulty = _CEFR_TO_DIFFICULTY[cefr]
            examples   = [q for q in EXAMPLE_QUESTIONS if q.cefr == cefr]
            print(f'  {cefr} ({difficulty}) — {subtopic_req.subtopic} — type: {_CEFR_QUESTION_TYPE[cefr]}')
            agent.forward(
                schema=schema, example_questions=examples,
                topic=schema.topic, subtopic=subtopic_req.subtopic,
                target_cefr=cefr, target_difficulty=difficulty,
                target_count=store.count(difficulty) + target_count,
                rejected=rejected, warnings=warnings,
            )
    return store, rejected, warnings

In [ ]:
SCHEMA = InputSchema(
    topic='Reading Real-World Notices',
    subtopics=[
        SubtopicRequirement(subtopic='public places',      a1_count=1, a2_count=1),
        SubtopicRequirement(subtopic='health and safety',  b1_count=1, b2_count=1),
        SubtopicRequirement(subtopic='academic settings',  c1_count=1, c2_count=1),
    ],
    constraints=GenerationConstraints(questions_per_iteration=3, max_iterations=8, rubric_min_score=0.65),
)

print('Schema:', SCHEMA.topic)
for s in SCHEMA.subtopics:
    print(f'  {s.subtopic}: total={s.total}')

store, rejected, warnings = run_image_t2t_generation(SCHEMA)
print(f'\nGenerated: easy={len(store.easy)} medium={len(store.medium)} hard={len(store.hard)}')
print(f'Rejected : {len(rejected)}')

In [ ]:
for difficulty in ('easy', 'medium', 'hard'):
    items = getattr(store, difficulty)
    if not items: continue
    print(f'\n{"="*70}  {difficulty.upper()}')
    for item in items:
        print(f'Q{item.question_number} [{item.target_cefr}] {item.question_type} — {item.subtopic}')
        print(f'  Notice: {item.image_content}')
        print(f'  {item.instruction}')
        print(f'  Q: {item.question}')
        print(f'  A: {item.expected_answer}')
        print()

In [ ]:
output = {
    'metadata': {
        'topic': SCHEMA.topic,
        'question_type': 'image_t2t',
        'total': store.total,
        'rejected': len(rejected),
    },
    'questions': {
        'easy':   [q.model_dump() for q in store.easy],
        'medium': [q.model_dump() for q in store.medium],
        'hard':   [q.model_dump() for q in store.hard],
    },
    'rejected': rejected,
}
OUTPUT_PATH.write_text(json.dumps(output, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Saved {store.total} questions to {OUTPUT_PATH}')